In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.metrics import confusion_matrix, classification_report
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers

In [ ]:
# Set plot style
sns.set_theme(style="whitegrid")

In [ ]:

#  LOAD & COMBINE TRAIN/TEST DATA

IMAGE_DIR = "/kaggle/input/datasets/jangedoo/utkface-new/UTKFace" 
IMG_SIZE = (128, 128)

def parse_filename(filename):
    parts = filename.split('_')
    if len(parts) >= 4:
        try:
            age = int(parts[0])
            race_code = int(parts[2])
            
            if race_code == 0:
                nat = 0  # United States
            elif race_code == 1:
                nat = 1  # African
            elif race_code == 3:
                nat = 2  # Indian
            else:
                nat = 3  # Other
            return age, nat
        except ValueError:
            return None, None
    return None, None

file_records = []
for fname in os.listdir(IMAGE_DIR):
    if fname.endswith(".jpg"):
        age, nat = parse_filename(fname)
        if age is not None and nat is not None:
            file_records.append({'filepath': os.path.join(IMAGE_DIR, fname), 'age': age, 'nationality': nat})

df = pd.DataFrame(file_records)
nat_labels = {0: 'United States', 1: 'African', 2: 'Indian', 3: 'Other'}
df['nationality_name'] = df['nationality'].map(nat_labels)

In [ ]:
#  CLASS BALANCE CHECK

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Before Balancing
sns.countplot(data=df, x='nationality_name', ax=axes[0], palette='magma')
axes[0].set_title("Nationality Distribution (Before Balancing)", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Nationality Category")
axes[0].set_ylabel("Image Count")

# Resampling Data
target_samples = df['nationality'].value_counts().min()
balanced_dfs = []
for nat_class in df['nationality'].unique():
    class_df = df[df['nationality'] == nat_class]
    resampled_class = resample(class_df, replace=(len(class_df) < target_samples), n_samples=target_samples, random_state=42)
    balanced_dfs.append(resampled_class)

df_balanced = pd.concat(balanced_dfs).sample(frac=1, random_state=42).reset_index(drop=True)

# After Balancing
sns.countplot(data=df_balanced, x='nationality_name', ax=axes[1], palette='viridis')
axes[1].set_title("Nationality Distribution (After Balancing)", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Nationality Category")
axes[1].set_ylabel("Image Count")

plt.tight_layout()
plt.savefig("dataset_balancing_visualization.png")
plt.show()

In [ ]:
#  PREPROCESS & SPLIT DATA
X_images, y_nat, y_age = [], [], []

for idx, row in df_balanced.iterrows():
    img = cv2.imread(row['filepath'])
    if img is not None:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, IMG_SIZE) / 255.0
        X_images.append(img)
        y_nat.append(row['nationality'])
        y_age.append(row['age'])

X_images = np.array(X_images, dtype='float32')
y_nat = tf.keras.utils.to_categorical(np.array(y_nat), num_classes=4)
y_age = np.array(y_age, dtype='float32')

X_train, X_test, y_nat_train, y_nat_test, y_age_train, y_age_test = train_test_split(
    X_images, y_nat, y_age, test_size=0.20, random_state=42
)

In [ ]:

#  BUILD  MULTI-TASK CNN

def build_multitask_cnn():
    inputs = layers.Input(shape=(128, 128, 3))
    
    
    x = layers.RandomFlip("horizontal")(inputs)
    x = layers.RandomRotation(0.1)(x)
    x = layers.RandomZoom(0.1)(x)
    x = layers.RandomContrast(0.1)(x)
    
  
    x = layers.Conv2D(32, (3, 3), activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.2)(x)
    
    
    x = layers.Conv2D(64, (3, 3), activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.3)(x)
    
    
    x = layers.Conv2D(128, (3, 3), activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.4)(x)
    
    
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    
    
    nationality_out = layers.Dense(4, activation='softmax', name='nationality_output')(x)
    age_out = layers.Dense(1, activation='linear', name='age_output')(x)
    
    return models.Model(inputs=inputs, outputs=[nationality_out, age_out])

model = build_multitask_cnn()

#  Balanced Loss Configuration 
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss={
        'nationality_output': 'categorical_crossentropy', 
        'age_output': tf.keras.losses.Huber()  # Robust to age outliers compared to MSE
    },
    loss_weights={
        'nationality_output': 1.0, 
        'age_output': 0.02  
    },
    metrics={
        'nationality_output': 'accuracy', 
        'age_output': 'mae'
    }
)

#  Callbacks to prevent over-training 
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', 
        patience=6, 
        restore_best_weights=True, 
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', 
        factor=0.5, 
        patience=3, 
        min_lr=1e-6, 
        verbose=1
    )
]



In [ ]:
# Train Model with Callbacks
history = model.fit(
    X_train, 
    {'nationality_output': y_nat_train, 'age_output': y_age_train},
    validation_data=(X_test, {'nationality_output': y_nat_test, 'age_output': y_age_test}),
    epochs=30,  
    batch_size=32,
    callbacks=callbacks
)

model.save("nationality_attribute_model.keras")

In [ ]:
model.save("nationality_attribute_model.keras")

In [ ]:
#  MODEL TRAINING CURVES
fig, ax = plt.subplots(1, 3, figsize=(18, 5))

# Loss Curves
ax[0].plot(history.history['loss'], label='Train Loss', color='blue')
ax[0].plot(history.history['val_loss'], label='Val Loss', color='orange')
ax[0].set_title("Total Model Loss", fontweight='bold')
ax[0].set_xlabel("Epochs")
ax[0].legend()

# Accuracy Curves
ax[1].plot(history.history['nationality_output_accuracy'], label='Train Accuracy', color='green')
ax[1].plot(history.history['val_nationality_output_accuracy'], label='Val Accuracy', color='red')
ax[1].set_title("Nationality Prediction Accuracy", fontweight='bold')
ax[1].set_xlabel("Epochs")
ax[1].legend()

# Age MAE Curves
ax[2].plot(history.history['age_output_mae'], label='Train Age MAE', color='purple')
ax[2].plot(history.history['val_age_output_mae'], label='Val Age MAE', color='brown')
ax[2].set_title("Age Estimation Absolute Error (MAE)", fontweight='bold')
ax[2].set_xlabel("Epochs")
ax[2].legend()

plt.tight_layout()
plt.savefig("model_training_curves.png")
plt.show()

In [ ]:
# CONFUSION MATRIX
y_nat_pred_probs, _ = model.predict(X_test)
y_nat_pred = np.argmax(y_nat_pred_probs, axis=1)
y_nat_true = np.argmax(y_nat_test, axis=1)

cm = confusion_matrix(y_nat_true, y_nat_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=nat_labels.values(), yticklabels=nat_labels.values())
plt.title("Confusion Matrix: Nationality Predictions", fontweight='bold')
plt.xlabel("Predicted Class")
plt.ylabel("True Class")
plt.tight_layout()
plt.savefig("nationality_confusion_matrix.png")
plt.show()